# 03 V2 - CUDA MLP baselines, controlled improvements, and final deployment

Uses shared.real_training and scripts/train_poseforge.py. No fixed dataset sizes or held-out person names. Baselines: Small, Base, Wide with ordinary CE. Improvements: source-aware square-root weights; smoothing 0.00/0.05/0.10; supported keypoint augmentation; source quota and mild balance. Only validated improvements may be combined. Top two candidates are confirmed at seeds 302, 303, 304. Final test evaluation has a persistent once-only guard.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path(os.environ['POSEFORGE_ROOT']).expanduser().resolve() if os.environ.get('POSEFORGE_ROOT') else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p/'shared').is_dir() and (p/'scripts').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get('POSEFORGE_DATA', ROOT/'artifacts/dataset_working/Real_Posture_Dataset'))
REPORTS = ROOT/'artifacts/reports'
RUN_VALUE = os.environ.get('POSEFORGE_RUN')
RUN = Path(RUN_VALUE).resolve() if RUN_VALUE else None
SEED = 302
DEVICE = os.environ.get('POSEFORGE_DEVICE', 'cuda:0')
# Optional explicit people; None uses a seeded, quality-eligible subject selection.
VAL_PEOPLE = os.environ.get('POSEFORGE_VAL_SUBJECTS', '').split() or None
TEST_PEOPLE = os.environ.get('POSEFORGE_TEST_SUBJECTS', '').split() or None

def stage(name, *extra):
    command = [sys.executable, str(ROOT/'scripts/train_poseforge.py'), '--stage', name,
               '--data', str(DATA), '--reports', str(REPORTS), '--seed', str(SEED)]
    if name not in ('audit', 'prepare'):
        assert RUN is not None, 'Set POSEFORGE_RUN to a prepared real_vNNN directory'
        command += ['--run', str(RUN), '--device', DEVICE, '--resume']
    subprocess.run(command + list(extra), cwd=ROOT, check=True)

def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
print('Python:',sys.executable,'\nRepository:',ROOT,'\nDataset:',DATA)


In [ ]:
assert RUN is not None, 'Set POSEFORGE_RUN explicitly'
import torch
from shared.real_training import DEFAULT, AUGMENTATION
print(torch.__version__,torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
print(DEFAULT)
RUN_FINAL=False  # Enable only when ready to freeze hyperparameters and consume final test.
split_info=read(RUN/'split_manifest.json')
print(split_info)

In [ ]:
stage('baseline')
display(pd.read_csv(RUN/'comparison.csv'))

In [ ]:
stage('improve')
comparison=pd.read_csv(RUN/'comparison.csv')
display(comparison)
display(pd.read_csv(RUN/'seed_confirmation.csv'))
display(pd.read_csv(RUN/'confirmed_ranking.csv'))
print('Frozen selection:',read(RUN/'selection.json'))

In [ ]:
from IPython.display import Image, display
selection=read(RUN/'selection.json')
chosen=RUN/'experiments'/selection['candidate']
display(Image(filename=str(chosen/'learning_curves.png')))
display(Image(filename=str(chosen/'confusion_matrix.png')))
display(pd.read_csv(chosen/'per_class_metrics.csv'))
display(pd.read_csv(chosen/'per_subject_metrics.csv'))
display(pd.read_csv(chosen/'camera_comparison.csv'))

In [ ]:
if RUN_FINAL:
    stage('finalize')
    stage('export')
    model_dir=Path(read(RUN/'final_model.json')['path'])
    print('Installed final model:',model_dir)
    print('Held-out test:',read(model_dir/'test_metrics.json'))
    print('Real-feature ONNX parity:',read(model_dir/'onnx_real_parity.json'))
else:
    print('Validation complete. Final test remains untouched by this notebook.')

Training logs and resumable optimizer/RNG checkpoints are saved for every experiment. Completed runs are skipped only when their configuration matches. The final PT checkpoint remains directly usable by inference/run.py; ONNX export checks real train/validation features without repeating test evaluation. No private videos are packaged.